# JPM Systematic Relative Value Backtester — USD SOFR Swap Curve

**Strategy:** OLS regression-based RV with 12 trigger combination grid search, quarterly P&L decomposition, category attribution, and traffic light regime filter.

**Reference:** J.P. Morgan — "RV on the EUR swap yield curve" (Gupta, Bassi, Vijay — Apr 2021)

In [ ]:
import datetime
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.regression_rv import RegressionRVConfig, rolling_regression
from BT.signals.regime_filter import RegimeFilterConfig, traffic_light
from BT.signals.rv_backtest import RVBacktestConfig, RVBacktestResult, run_rv_backtest

In [ ]:
config = {
    # === DATA ===
    "data_start": "2021-01-01",
    "data_end": None,
    "mtm_mode": "approximate",
    # === FLY UNIVERSE ===
    "fly_categories": {
        "standard_spot": [
            ("2Y","5Y","10Y"), ("2Y","5Y","7Y"), ("3Y","5Y","10Y"),
            ("2Y","10Y","30Y"), ("5Y","10Y","30Y"), ("5Y","7Y","10Y"),
            ("10Y","20Y","30Y"),
        ],
        "5y_gap_fwd": [
            ("5Yx5Y","10Yx5Y","15Yx5Y"),
            ("10Yx5Y","15Yx5Y","20Yx5Y"),
            ("15Yx5Y","20Yx5Y","25Yx5Y"),
        ],
        "1y_gap_fwd": [
            ("1Yx1Y","2Yx1Y","3Yx1Y"),
            ("2Yx1Y","3Yx1Y","4Yx1Y"),
            ("3Yx1Y","4Yx1Y","5Yx1Y"),
        ],
        "2y_gap_fwd": [
            ("2Yx2Y","4Yx2Y","6Yx2Y"),
        ],
    },
    "enabled_categories": ["standard_spot", "1y_gap_fwd", "5y_gap_fwd"],
    # === REGRESSION ===
    "regression_window_days": 130,
    # === ENTRY TRIGGER GRID ===
    "entry_grid": {
        "r_squared": [0.60, 0.80],
        "residual_bp": [2.0, 3.0, 4.0],
        "zscore": [1.5, 2.0],
    },
    # === EXIT RULES ===
    "exit_mean_reversion": True,
    "exit_stop_loss_additional_sd": 2.0,
    "exit_max_holding_days": 22,
    # === PORTFOLIO RULES ===
    "no_duplicate_flies": True,
    "reentry_after_stop": True,
    # === TRAFFIC LIGHT ===
    "traffic_light_enabled": True,
    "traffic_light_reference_fly": ("1Yx1Y","2Yx1Y","3Yx1Y"),
    "beta_regression_window_days": 130,
    "beta_vol_window_days": 65,
    "beta_vol_zscore_window_days": 130,
    "traffic_light_threshold": 3.0,
    "traffic_light_threshold_options": [2.0, 2.5, 3.0, 3.5, 4.0],
    # === COSTS ===
    "round_trip_cost_bp": 0.5,
    "min_profit_to_cost_ratio": 2.0,
}

In [ ]:
curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
irswaps_tb = IRSwapsTB(curve_mdp, show_tqdm=True)
tb = TimeseriesBuilder(irswaps_tb=irswaps_tb)

end_date = config["data_end"] or datetime.date.today()
start_date = datetime.datetime.strptime(config["data_start"], "%Y-%m-%d").date()

# Collect all unique tenors across all fly categories
all_tenors = set()
for cat in config["enabled_categories"]:
    for fly_tuple in config["fly_categories"].get(cat, []):
        for t in fly_tuple:
            all_tenors.add(t)

# Build queries — parse "1Yx1Y" as fwd=1Y, tenor=1Y
queries = []
tenor_to_query = {}
for t in sorted(all_tenors):
    if "x" in t or "X" in t:
        # Forward notation: "5Yx5Y" -> fwd=5Y, tenor=5Y -> query tenor = "5Y5Y"
        parts = t.upper().split("X")
        query_tenor = f"{parts[0]}{parts[1]}"
    else:
        query_tenor = t
    q = IRSwapQuery(curve="USD-SOFR-1D", tenor=query_tenor, value=IRSwapValue.RATE)
    queries.append(q)
    tenor_to_query[t] = q

rates_df = tb.get_timeseries(start=start_date, end=end_date, queries=queries, n_jobs=8)

# Rename columns to match fly tenor labels
col_rename = {}
for t, q in tenor_to_query.items():
    col = q.col_name(cube_name="USD-SOFR-1D")
    if col in rates_df.columns:
        col_rename[col] = t
rates_df = rates_df.rename(columns=col_rename)

print(f"Loaded: {rates_df.shape[0]} dates x {rates_df.shape[1]} tenors")
print(f"Tenors: {sorted(rates_df.columns.tolist())}")

In [ ]:
reg_config = RegressionRVConfig(
    window_days=config["regression_window_days"],
    zscore_lookback_days=config["regression_window_days"],
)

all_residuals, all_zscores, all_rsq = {}, {}, {}
all_weights, all_rates = {}, {}
all_betas_body, all_betas_curve = {}, {}
fly_categories_map = {}

for cat in config["enabled_categories"]:
    for fly_tuple in config["fly_categories"].get(cat, []):
        left, belly, right = fly_tuple
        if not all(t in rates_df.columns for t in fly_tuple):
            print(f"Skipping {fly_tuple}: missing tenor data")
            continue

        fly_id = "/".join(fly_tuple)
        df3 = rates_df[[left, belly, right]].dropna()

        # 50:50 butterfly
        fly_50_50 = 0.5 * df3[left] + 0.5 * df3[right] - df3[belly]
        body = df3[belly]
        wing_curve = df3[right] - df3[left]

        reg_result = rolling_regression(fly_50_50, body, wing_curve, reg_config)

        all_residuals[fly_id] = reg_result.residuals
        all_zscores[fly_id] = reg_result.zscores
        all_rsq[fly_id] = reg_result.rsq
        all_betas_body[fly_id] = reg_result.betas_body
        all_betas_curve[fly_id] = reg_result.betas_curve

        # Weights from regression hedge ratios
        all_weights[fly_id] = pd.DataFrame({
            "left": 0.5 - reg_result.betas_curve,
            "belly": 1.0,
            "right": 0.5 + reg_result.betas_curve,
        }, index=df3.index)
        all_rates[fly_id] = df3.rename(columns={left: "left", belly: "belly", right: "right"})
        fly_categories_map[fly_id] = cat

print(f"Computed regression for {len(all_residuals)} flies")

## Traffic Light & Grid Search

In [ ]:
ref_fly_id = "/".join(config["traffic_light_reference_fly"])
if config["traffic_light_enabled"] and ref_fly_id in all_betas_body:
    tl_config = RegimeFilterConfig(
        beta_vol_window_days=config["beta_vol_window_days"],
        beta_vol_zscore_window_days=config["beta_vol_zscore_window_days"],
        threshold=config["traffic_light_threshold"],
    )
    tl_result = traffic_light(all_betas_body[ref_fly_id], all_betas_curve[ref_fly_id], tl_config)
    regime_series = tl_result["regime"]
    print(f"Traffic light: {(regime_series == 'red').sum()} red days, "
          f"{(regime_series == 'green').sum()} green days")
else:
    regime_series = pd.Series("green", index=rates_df.index)
    tl_result = None
    print("Traffic light disabled or reference fly not found")

In [ ]:
trigger_grid = list(itertools_product(
    config["entry_grid"]["r_squared"],
    config["entry_grid"]["residual_bp"],
    config["entry_grid"]["zscore"],
))

print(f"Running {len(trigger_grid)} trigger combinations...")
grid_results = []

for rsq_t, res_t, zs_t in trigger_grid:
    bt_config = RVBacktestConfig(
        mtm_mode=config["mtm_mode"],
        entry_min_rsq=rsq_t,
        entry_min_residual_bp=res_t,
        entry_min_zscore=zs_t,
        exit_mean_reversion=config["exit_mean_reversion"],
        exit_stop_loss_sd=config["exit_stop_loss_additional_sd"],
        exit_max_holding_days=config["exit_max_holding_days"],
        no_duplicate_flies=config["no_duplicate_flies"],
        round_trip_cost_bp=config["round_trip_cost_bp"],
        min_profit_to_cost_ratio=config["min_profit_to_cost_ratio"],
    )

    # With traffic light
    r_with = run_rv_backtest(
        all_residuals, all_zscores, all_rsq, all_weights, all_rates,
        regime_series, fly_categories_map, config=bt_config,
    )
    # Without traffic light
    regime_off = pd.Series("green", index=rates_df.index)
    r_without = run_rv_backtest(
        all_residuals, all_zscores, all_rsq, all_weights, all_rates,
        regime_off, fly_categories_map, config=bt_config,
    )

    grid_results.append({
        "R_sq": rsq_t, "Resid_bp": res_t, "Z_score": zs_t,
        # With TL
        "n_trades_tl": r_with.metrics["n_trades"],
        "hit_rate_tl": r_with.metrics["hit_rate"],
        "avg_pnl_tl": r_with.metrics["avg_pnl"],
        "sharpe_tl": r_with.metrics["sharpe"],
        "total_pnl_tl": r_with.metrics["total_pnl"],
        "max_dd_tl": r_with.metrics["max_drawdown"],
        # Without TL
        "n_trades_no_tl": r_without.metrics["n_trades"],
        "hit_rate_no_tl": r_without.metrics["hit_rate"],
        "avg_pnl_no_tl": r_without.metrics["avg_pnl"],
        "sharpe_no_tl": r_without.metrics["sharpe"],
        "total_pnl_no_tl": r_without.metrics["total_pnl"],
        # Store result objects for best config
        "_result_with": r_with,
        "_result_without": r_without,
    })

grid_df = pd.DataFrame(grid_results)
display(grid_df[["R_sq", "Resid_bp", "Z_score",
    "n_trades_tl", "hit_rate_tl", "avg_pnl_tl", "sharpe_tl", "total_pnl_tl",
    "n_trades_no_tl", "hit_rate_no_tl", "sharpe_no_tl"]].style.format({
    "hit_rate_tl": "{:.1%}", "avg_pnl_tl": "{:.2f}", "sharpe_tl": "{:.2f}",
    "total_pnl_tl": "{:.1f}", "hit_rate_no_tl": "{:.1%}", "sharpe_no_tl": "{:.2f}",
}))

# Select best trigger combo by Sharpe
best_idx = grid_df["sharpe_tl"].idxmax()
best = grid_df.loc[best_idx]
print(f"\nBest trigger: R_sq={best['R_sq']}, Resid={best['Resid_bp']}bp, Z={best['Z_score']} "
      f"-> Sharpe={best['sharpe_tl']:.2f}, Hit={best['hit_rate_tl']:.1%}")
best_result = best["_result_with"]

## P&L Analytics & Attribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
best_result.cumulative_pnl.plot(ax=ax, label="Total", linewidth=2, color="black")
for cat in best_result.daily_pnl_by_category.columns:
    best_result.daily_pnl_by_category[cat].cumsum().plot(ax=ax, label=cat, alpha=0.7)

# Shade red regime periods
if tl_result is not None:
    red_mask = tl_result["regime"] == "red"
    in_red = False
    for dt, val in red_mask.items():
        if val and not in_red:
            start = dt; in_red = True
        elif not val and in_red:
            ax.axvspan(start, dt, alpha=0.1, color="red"); in_red = False

ax.set_title(f"Cumulative P&L (R_sq>={best['R_sq']}, Resid>={best['Resid_bp']}bp, Z>={best['Z_score']})")
ax.legend(); ax.grid(True, alpha=0.3)

dd = best_result.cumulative_pnl - best_result.cumulative_pnl.cummax()
axes[1].fill_between(dd.index, dd.values, 0, alpha=0.5, color="red")
axes[1].set_title("Drawdown (bp)"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
quarterly = best_result.daily_pnl.resample("QE").sum()
colors = ["green" if x >= 0 else "red" for x in quarterly.values]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(quarterly.index, quarterly.values, width=60, color=colors, alpha=0.7)
ax.set_title("Quarterly P&L (bp)")
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
trade_df = pd.DataFrame([{
    "fly_id": t.fly_id, "category": t.category,
    "entry_date": t.entry_date, "exit_date": t.exit_date,
    "exit_reason": t.exit_reason, "entry_zscore": t.entry_zscore,
    "pnl_bp": t.realized_pnl,
    "holding_days": (len(pd.bdate_range(t.entry_date, t.exit_date)) - 1) if t.exit_date else 0,
} for t in best_result.trades])

if len(trade_df) > 0:
    cat_stats = trade_df.groupby("category")["pnl_bp"].agg(
        ["count", "mean", "std", "sum",
         lambda x: (x > 0).mean()]
    ).rename(columns={"<lambda_0>": "hit_rate"})
    display(cat_stats.style.format({
        "mean": "{:.2f}", "std": "{:.2f}", "sum": "{:.1f}", "hit_rate": "{:.1%}",
    }))

In [ ]:
tl_impact = []
for _, row in grid_df.iterrows():
    tl_impact.append({
        "R_sq": row["R_sq"], "Resid_bp": row["Resid_bp"], "Z_score": row["Z_score"],
        "Sharpe_with_TL": row["sharpe_tl"],
        "Sharpe_without_TL": row["sharpe_no_tl"],
        "Sharpe_delta": row["sharpe_tl"] - row["sharpe_no_tl"],
        "HR_with_TL": row["hit_rate_tl"],
        "HR_without_TL": row["hit_rate_no_tl"],
        "HR_delta": row["hit_rate_tl"] - row["hit_rate_no_tl"],
        "PnL_with_TL": row["total_pnl_tl"],
        "PnL_without_TL": row["total_pnl_no_tl"],
    })

tl_impact_df = pd.DataFrame(tl_impact)
display(tl_impact_df.style.format({
    "Sharpe_with_TL": "{:.2f}", "Sharpe_without_TL": "{:.2f}", "Sharpe_delta": "{:+.2f}",
    "HR_with_TL": "{:.1%}", "HR_without_TL": "{:.1%}", "HR_delta": "{:+.1%}",
    "PnL_with_TL": "{:.1f}", "PnL_without_TL": "{:.1f}",
}))

In [ ]:
if tl_result is not None:
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.plot(best_result.cumulative_pnl.index, best_result.cumulative_pnl.values,
             color="black", linewidth=1.5, label="Cumulative P&L")
    ax1.set_ylabel("P&L (bp)")

    ax2 = ax1.twinx()
    ax2.plot(tl_result["indicator"].index, tl_result["indicator"].values,
             color="orange", alpha=0.5, linewidth=0.8, label="Traffic Light")
    ax2.axhline(y=config["traffic_light_threshold"], color="red", linestyle="--", alpha=0.5)
    ax2.set_ylabel("Traffic Light Indicator")

    # Shade red periods
    red_mask = tl_result["regime"] == "red"
    in_red = False
    for dt, val in red_mask.items():
        if val and not in_red:
            start = dt; in_red = True
        elif not val and in_red:
            ax1.axvspan(start, dt, alpha=0.15, color="red"); in_red = False

    ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
    ax1.set_title("Cumulative P&L with Traffic Light Overlay")
    plt.tight_layout(); plt.show()

## Diagnostics & Validation

In [ ]:
if ref_fly_id in all_betas_body:
    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

    # Beta vs body
    all_betas_body[ref_fly_id].dropna().plot(ax=axes[0], color="blue", linewidth=0.8)
    axes[0].set_title(f"Rolling beta_body — {ref_fly_id}")
    axes[0].grid(True, alpha=0.3)

    # Beta vs curve
    all_betas_curve[ref_fly_id].dropna().plot(ax=axes[1], color="green", linewidth=0.8)
    axes[1].set_title(f"Rolling beta_curve — {ref_fly_id}")
    axes[1].grid(True, alpha=0.3)

    # Beta vol
    if tl_result is not None:
        vol_body = all_betas_body[ref_fly_id].rolling(config["beta_vol_window_days"]).std()
        vol_curve = all_betas_curve[ref_fly_id].rolling(config["beta_vol_window_days"]).std()
        vol_body.dropna().plot(ax=axes[2], color="blue", linewidth=0.8, label="beta_body vol")
        vol_curve.dropna().plot(ax=axes[2], color="green", linewidth=0.8, label="beta_curve vol")
        axes[2].set_title("3M Rolling Beta Volatility"); axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        tl_result["indicator"].dropna().plot(ax=axes[3], color="orange", linewidth=0.8)
        axes[3].axhline(y=config["traffic_light_threshold"], color="red", linestyle="--")
        axes[3].set_title("Traffic Light Indicator"); axes[3].grid(True, alpha=0.3)

    # Shade red periods on all
    if tl_result is not None:
        red_mask = tl_result["regime"] == "red"
        for ax in axes:
            in_red = False
            for dt, val in red_mask.items():
                if val and not in_red:
                    start = dt; in_red = True
                elif not val and in_red:
                    ax.axvspan(start, dt, alpha=0.1, color="red"); in_red = False

    plt.tight_layout(); plt.show()

In [ ]:
if len(trade_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for cat in trade_df["category"].unique():
        mask = trade_df["category"] == cat
        axes[0].scatter(trade_df.loc[mask, "entry_zscore"], trade_df.loc[mask, "pnl_bp"],
                       s=15, alpha=0.5, label=cat)
    axes[0].axhline(y=0, color="black", linewidth=0.5)
    axes[0].set_xlabel("Entry Z-score"); axes[0].set_ylabel("P&L (bp)")
    axes[0].set_title("P&L vs Entry Z-score"); axes[0].legend(fontsize=8)

    for cat in trade_df["category"].unique():
        mask = trade_df["category"] == cat
        axes[1].scatter(trade_df.loc[mask, "holding_days"], trade_df.loc[mask, "pnl_bp"],
                       s=15, alpha=0.5, label=cat)
    axes[1].axhline(y=0, color="black", linewidth=0.5)
    axes[1].set_xlabel("Holding Period (days)"); axes[1].set_ylabel("P&L (bp)")
    axes[1].set_title("P&L vs Holding Period"); axes[1].legend(fontsize=8)

    plt.tight_layout(); plt.show()

In [ ]:
if config["traffic_light_enabled"]:
    tl_sensitivity = []
    for threshold in config["traffic_light_threshold_options"]:
        tl_cfg = RegimeFilterConfig(
            beta_vol_window_days=config["beta_vol_window_days"],
            beta_vol_zscore_window_days=config["beta_vol_zscore_window_days"],
            threshold=threshold,
        )
        tl_r = traffic_light(all_betas_body[ref_fly_id], all_betas_curve[ref_fly_id], tl_cfg)
        regime = tl_r["regime"]

        # Run best trigger combo with this threshold
        bt_cfg = RVBacktestConfig(
            mtm_mode=config["mtm_mode"],
            entry_min_rsq=best["R_sq"], entry_min_residual_bp=best["Resid_bp"],
            entry_min_zscore=best["Z_score"],
            exit_mean_reversion=config["exit_mean_reversion"],
            exit_stop_loss_sd=config["exit_stop_loss_additional_sd"],
            exit_max_holding_days=config["exit_max_holding_days"],
            round_trip_cost_bp=config["round_trip_cost_bp"],
        )
        r = run_rv_backtest(
            all_residuals, all_zscores, all_rsq, all_weights, all_rates,
            regime, fly_categories_map, config=bt_cfg,
        )
        red_days = (regime == "red").sum()
        tl_sensitivity.append({
            "threshold": threshold,
            "red_days": red_days,
            "n_trades": r.metrics["n_trades"],
            "hit_rate": r.metrics["hit_rate"],
            "sharpe": r.metrics["sharpe"],
            "total_pnl": r.metrics["total_pnl"],
        })

    tl_sens_df = pd.DataFrame(tl_sensitivity)
    display(tl_sens_df.style.format({
        "hit_rate": "{:.1%}", "sharpe": "{:.2f}", "total_pnl": "{:.1f}",
    }))

In [ ]:
# Quick validation: compare in-sample vs out-of-sample residual variance
if ref_fly_id in all_residuals:
    oos_resid = all_residuals[ref_fly_id].dropna()
    # In-sample: refit on entire window and compute residual (should be tighter)
    ref_tenors = config["traffic_light_reference_fly"]
    df3 = rates_df[[ref_tenors[0], ref_tenors[1], ref_tenors[2]]].dropna()
    fly_50 = 0.5 * df3.iloc[:, 0] + 0.5 * df3.iloc[:, 2] - df3.iloc[:, 1]
    body = df3.iloc[:, 1]
    curve_spread = df3.iloc[:, 2] - df3.iloc[:, 0]

    from sklearn.linear_model import LinearRegression
    X_full = np.column_stack([body.values, curve_spread.values])
    lr = LinearRegression().fit(X_full, fly_50.values)
    in_sample_resid = fly_50.values - lr.predict(X_full)

    print(f"In-sample residual std: {np.std(in_sample_resid)*10000:.2f} bp")
    print(f"OOS residual std:       {oos_resid.std()*10000:.2f} bp")
    assert oos_resid.std() >= np.std(in_sample_resid) * 0.9, "OOS should be >= in-sample variance"